In [0]:
 dbutils.widgets.dropdown(
    name="environment",
    defaultValue="dev",
    choices=["dev","prod","qa"],
    label="select Environment"
)
env = dbutils.widgets.get("environment")

goldTablName   = f"saleslake_{env}.gold_{env}.tgtInvoice"
silverTablName = f"saleslake_{env}.silver_{env}.cleaninvoice"
print(f"silver: {silverTablName}\ngold  : {goldTablName}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Load silver and gold tables
silver_df = spark.table(silverTablName)
gold_dt   = DeltaTable.forName(spark, goldTablName)

# 1. latest_inv_silver → incremental load
max_start_ts = gold_dt.toDF().agg(
    F.coalesce(F.max("start_effective_ts"), F.to_timestamp(F.lit("1990-01-01"), "yyyy-MM-dd"))
).collect()[0][0]

latest_inv_silver = silver_df.filter(F.col("ingest_ts") > max_start_ts)

# 2. latest_rm_dup_silver → remove duplicates, keep latest per invoice_id
window_spec = Window.partitionBy("invoice_id").orderBy(F.col("ingest_ts").desc())
latest_rm_dup_silver = (
    latest_inv_silver
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# 3. silver_gold_rec → join with active gold records
active_gold = gold_dt.toDF().filter(F.col("is_active") == "Y")

silver_gold_rec = (
    latest_rm_dup_silver.alias("s")
    .join(active_gold.alias("g"), F.col("s.invoice_id") == F.col("g.invoice_id"), "left")
    .select(
        "s.*",
        "g.customer_sk", "g.is_active", "g.rec_version",
        "g.start_effective_ts", "g.end_effective_ts",
        F.when(F.col("g.customer_sk").isNull(), F.lit(1))
         .otherwise(F.col("g.rec_version") + 1).alias("new_rec_version"),
        F.when(F.col("g.customer_sk").isNull(), F.lit("NEW"))
         .when(
             (F.col("s.subtotal_amount") != F.col("g.subtotal_amount")) |
             (F.col("s.discount_code")   != F.col("g.discount_code"))   |
             (F.col("s.discount_amount") != F.col("g.discount_amount")) |
             (F.col("s.tax_amount")      != F.col("g.tax_amount"))      |
             (F.col("s.total_amount")    != F.col("g.total_amount"))    |
             (F.col("s.payment_status")  != F.col("g.payment_status"))  |
             (F.col("s.payment_method")  != F.col("g.payment_method"))  |
             (F.col("s.payment_date")    != F.col("g.payment_date"))    |
             (F.col("s.due_date")        != F.col("g.due_date"))        |
             (F.col("s.customer_id")     != F.col("g.customer_id"))     |
             (F.col("s.region")          != F.col("g.region"))          |
             (F.col("s.store_id")        != F.col("g.store_id"))        |
             (F.col("s.channel")         != F.col("g.channel")),
             F.lit("CHANGE")
         )
         .otherwise(F.lit("NO_CHANGE")).alias("rec_flag")
    )
)

# 4. insert_flag and changed_flag
insert_flag = silver_gold_rec.filter(F.col("rec_flag").isin("NEW", "CHANGE")) \
    .withColumn("merge_flag", F.lit("INSERT")) \
    .withColumn("inv_merge_key", F.lit(None))

changed_flag = silver_gold_rec.filter(F.col("rec_flag") == "CHANGE") \
    .withColumn("merge_flag", F.lit("UPDATE")) \
    .withColumn("inv_merge_key", F.col("invoice_id"))

src_df = insert_flag.unionByName(changed_flag)

# 5. Perform MERGE into gold table
(
    gold_dt.alias("tgt")
    .merge(src_df.alias("src"), "tgt.invoice_id = src.inv_merge_key")
    .whenMatchedUpdate(
        condition="src.merge_flag = 'UPDATE'",
        set={
            "is_active": F.lit("N"),
            "end_effective_ts": F.current_timestamp()
        }
    )
    .whenNotMatchedInsert(
        condition="src.merge_flag = 'INSERT'",
        values={
            "invoice_id": "src.invoice_id",
            "invoice_number": "src.invoice_number",
            "customer_id": "src.customer_id",
            "invoice_date": "src.invoice_date",
            "due_date": "src.due_date",
            "subtotal_amount": "src.subtotal_amount",
            "discount_code": "src.discount_code",
            "discount_amount": "src.discount_amount",
            "tax_amount": "src.tax_amount",
            "total_amount": "src.total_amount",
            "payment_status": "src.payment_status",
            "payment_method": "src.payment_method",
            "payment_date": "src.payment_date",
            "currency": "src.currency",
            "region": "src.region",
            "store_id": "src.store_id",
            "channel": "src.channel",
            "created_by": "src.created_by",
            "is_active": F.lit("Y"),
            "rec_version": "src.new_rec_version",
            "start_effective_ts": F.current_timestamp(),
            "end_effective_ts": F.to_timestamp(F.lit("9999-12-31"), "yyyy-MM-dd")
        }
    )
    .execute()
)


In [0]:
# spark.sql(f"""
# MERGE INTO {goldTablName} tgt
# USING (
#     WITH latest_inv_silver AS (
#         SELECT *
#         FROM {silverTablName}
#         WHERE ingest_ts > (
#             SELECT COALESCE(MAX(start_effective_ts), TO_TIMESTAMP('1990-01-01','yyyy-MM-dd'))
#             FROM {goldTablName}
#         )
#     ),
#     latest_rm_dup_silver AS (
#         SELECT * FROM (
#             SELECT *,
#                    ROW_NUMBER() OVER (PARTITION BY invoice_id ORDER BY ingest_ts DESC) AS rn
#             FROM latest_inv_silver
#         ) WHERE rn = 1
#     ),  
#     silver_gold_rec AS (  
#         SELECT
#             s.*,
#             g.customer_sk          AS customer_sk,
#             g.is_active            AS is_active,
#             g.rec_version          AS rec_version,
#             g.start_effective_ts   AS start_effective_ts,
#             g.end_effective_ts     AS end_effective_ts,
#             CASE WHEN g.customer_sk IS NULL THEN 1 ELSE g.rec_version + 1 END AS new_rec_version,
#             CASE
#                 WHEN g.customer_sk IS NULL THEN 'NEW'
#                 WHEN NOT (s.subtotal_amount  <=> g.subtotal_amount)
#                   OR NOT (s.discount_code    <=> g.discount_code)
#                   OR NOT (s.discount_amount  <=> g.discount_amount)
#                   OR NOT (s.tax_amount       <=> g.tax_amount)
#                   OR NOT (s.total_amount     <=> g.total_amount)
#                   OR NOT (s.payment_status   <=> g.payment_status)
#                   OR NOT (s.payment_method   <=> g.payment_method)
#                   OR NOT (s.payment_date     <=> g.payment_date)
#                   OR NOT (s.due_date         <=> g.due_date)
#                   OR NOT (s.customer_id      <=> g.customer_id)
#                   OR NOT (s.region           <=> g.region)
#                   OR NOT (s.store_id         <=> g.store_id)
#                   OR NOT (s.channel          <=> g.channel)
#                 THEN 'CHANGE'
#                 ELSE 'NO_CHANGE'
#             END AS rec_flag
#         FROM latest_rm_dup_silver s
#         LEFT JOIN (SELECT * FROM {goldTablName} WHERE is_active = 'Y') g
#                ON s.invoice_id = g.invoice_id
#     ),
#     insert_flag AS (
#         SELECT
#             NULL AS inv_merge_key,
#             invoice_id, invoice_number, customer_id, invoice_date, due_date,
#             subtotal_amount, discount_code, discount_amount, tax_amount, total_amount,
#             payment_status, payment_method, payment_date,
#             currency, region, store_id, channel, created_by,
#             customer_sk, is_active, rec_version, start_effective_ts, end_effective_ts,
#             new_rec_version, rec_flag,
#             'INSERT' AS merge_flag
#         FROM silver_gold_rec
#         WHERE rec_flag IN ('NEW','CHANGE')
#     ),
#     changed_flag AS (
#         SELECT
#             invoice_id AS inv_merge_key,
#             invoice_id, invoice_number, customer_id, invoice_date, due_date,
#             subtotal_amount, discount_code, discount_amount, tax_amount, total_amount,
#             payment_status, payment_method, payment_date,
#             currency, region, store_id, channel, created_by,
#             customer_sk, is_active, rec_version, start_effective_ts, end_effective_ts,
#             new_rec_version, rec_flag,
#             'UPDATE' AS merge_flag
#         FROM silver_gold_rec
#         WHERE rec_flag = 'CHANGE'
#     )
#     SELECT * FROM changed_flag
#     UNION ALL
#     SELECT * FROM insert_flag
# ) src
# ON tgt.invoice_id = src.inv_merge_key

# WHEN MATCHED AND src.merge_flag = 'UPDATE' THEN UPDATE SET
#     is_active        = 'N',
#     end_effective_ts = current_timestamp()

# WHEN NOT MATCHED AND src.merge_flag = 'INSERT' THEN INSERT (
#     invoice_id, invoice_number, customer_id, invoice_date, due_date,
#     subtotal_amount, discount_code, discount_amount, tax_amount, total_amount,
#     payment_status, payment_method, payment_date,
#     currency, region, store_id, channel, created_by,
#     is_active, rec_version, start_effective_ts, end_effective_ts
# )
# VALUES (
#     invoice_id, invoice_number, customer_id, invoice_date, due_date,
#     subtotal_amount, discount_code, discount_amount, tax_amount, total_amount,
#     payment_status, payment_method, payment_date,
#     currency, region, store_id, channel, created_by,
#     'Y', new_rec_version, current_timestamp(), TO_TIMESTAMP('9999-12-31','yyyy-MM-dd')
# )
# """)

In [0]:
%sql
-- SELECT * FROM saleslake_{env}.gold_{env}.tgtinvoice';
-- SELECT * FROM saleslake_qa.silver_qa.cleaninvoice;

In [0]:
%sql
-- describe extended saleslake_dev.gold_dev.refinedinvoice;
-- describe extended saleslake_dev.silver_dev.cleanedinvoice;

In [0]:
%sql
-- describe extended saleslake_dev.gold_dev.refinedinvoice;

In [0]:
%sql
-- SELECT * FROM  saleslake_dev.gold_dev.refinedinvoice; 